# EduPulse — 전체 파이프라인 End-to-End 테스트

데이터 생성 → 전처리 → 학습 → 예측까지 전체 파이프라인을 단계별로 검증합니다.

**검증 단계:**
1. 합성 데이터 생성
2. 전처리 및 병합
3. 모델 학습 및 저장
4. 모델 로딩 및 예측
5. API 입력 시뮬레이션

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import os

import warnings
warnings.filterwarnings('ignore')

## Step 1: 합성 데이터 생성

In [ ]:
from edupulse.data.generators.run_all import run

print('합성 데이터 생성...')
run(n_years=8, start_year=2018)

# 생성 확인
for path in [
    '../edupulse/data/raw/internal/enrollment_history.csv',
    '../edupulse/data/raw/external/search_trends.csv',
    '../edupulse/data/raw/external/job_postings.csv',
]:
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    status = f'OK ({size:,} bytes)' if exists else 'MISSING'
    print(f'  {os.path.basename(path):30s}: {status}')

print('\nStep 1 완료')

## Step 2: 전처리 및 병합

In [ ]:
from edupulse.preprocessing.merger import build_training_dataset

print('학습 데이터셋 빌드...')
df = build_training_dataset()

print(f'\n데이터 shape: {df.shape}')
print(f'컬럼: {list(df.columns)}')
print(f'결측치:\n{df.isnull().sum()[df.isnull().sum() > 0]}')

# 필수 컬럼 검증
from edupulse.model.xgboost_model import FEATURE_COLUMNS, TARGET_COLUMN

required = FEATURE_COLUMNS + [TARGET_COLUMN]
for col in required:
    status = 'OK' if col in df.columns else 'MISSING'
    print(f'  {col:25s}: {status}')

print('\nStep 2 완료')

## Step 3: 모델 학습 및 저장

In [ ]:
# XGBoost 학습 + 저장
from edupulse.model.xgboost_model import XGBoostForecaster

xgb = XGBoostForecaster()
xgb.train(df)
xgb.save('../edupulse/model/saved/xgboost', version=1)

xgb_eval = xgb.evaluate(df, n_splits=5)
print(f'XGBoost 학습 완료 — MAPE: {xgb_eval["mape"]:.2f}%')
print(f'  저장: edupulse/model/saved/xgboost/v1/model.joblib')

In [ ]:
# Prophet 학습 + 저장 (선택적)
try:
    from edupulse.model.prophet_model import ProphetForecaster
    
    prophet = ProphetForecaster()
    prophet.train(df)
    prophet.save('../edupulse/model/saved/prophet', version=1)
    
    prophet_eval = prophet.evaluate(df, n_splits=5)
    print(f'Prophet 학습 완료 — MAPE: {prophet_eval["mape"]:.2f}%')
except ImportError:
    print('Prophet 미설치 — 건너뜀')

In [ ]:
# LSTM 학습 + 저장 (선택적)
try:
    from edupulse.model.lstm_model import LSTMForecaster
    
    lstm = LSTMForecaster()
    lstm.train(df, epochs=50)  # 테스트용으로 에포크 줄임
    lstm.save('../edupulse/model/saved/lstm', version=1)
    
    lstm_eval = lstm.evaluate(df, n_splits=3)
    print(f'LSTM 학습 완료 — MAPE: {lstm_eval["mape"]:.2f}%')
except ImportError:
    print('PyTorch 미설치 — 건너뜀')

print('\nStep 3 완료')

## Step 4: 모델 로딩 및 예측

In [ ]:
# 저장된 모델 로딩 테스트
xgb_loaded = XGBoostForecaster()
xgb_loaded.load('../edupulse/model/saved/xgboost', version=1)
print('XGBoost 로딩 성공')

# 테스트 예측
test_features = df[FEATURE_COLUMNS].iloc[-1:].fillna(0)
result = xgb_loaded.predict(test_features)

print(f'\n예측 결과:')
print(f'  예상 등록 인원: {result.predicted_enrollment}명')
print(f'  수요 등급: {result.demand_tier.value}')
print(f'  신뢰 구간: [{result.confidence_lower}, {result.confidence_upper}]')
print(f'  모델: {result.model_used}')
print(f'  MAPE: {result.mape:.2f}%' if result.mape else '  MAPE: N/A')

## Step 5: API 입력 시뮬레이션

In [ ]:
from edupulse.model.predict import predict_demand

# 다양한 시나리오 테스트
test_cases = [
    ('Python 웹개발', '2026-03-01', 'coding'),
    ('정보보안 실무', '2026-07-01', 'security'),
    ('Unity 게임개발', '2026-01-15', 'game'),
    ('디지털 일러스트', '2026-09-01', 'art'),
]

print(f'{"과정명":20s} {"시작일":12s} {"분야":10s} {"예측":>6s} {"등급":>6s} {"신뢰구간"}')
print('-' * 80)

for course, date, field in test_cases:
    # predict_demand는 ensemble을 기본으로 사용하나, 여기선 xgboost 직접 테스트
    result = predict_demand(course, date, field, model_name='xgboost')
    print(
        f'{course:20s} {date:12s} {field:10s} '
        f'{result.predicted_enrollment:>5d}명 '
        f'{result.demand_tier.value:>5s} '
        f'[{result.confidence_lower:.0f}–{result.confidence_upper:.0f}]'
    )

print('\n전체 파이프라인 테스트 완료')